# Feature Engineering (testing)

Exploration and sanity checks only. The dataset the pipeline trains on is built by the script, from the `research` folder:

```
..\.venv\Scripts\python -m modeling.build_features
```

Feature settings live in `modeling/config.py` (`FEATURE_SETTINGS`). Change them there so the script, the pipeline and inference stay in sync.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from modeling import features as F
from modeling.config import DATA_PATH, FEATURE_SETTINGS, RAW_PATH
from modeling.sheets import clean_raw, load_raw_data

In [ ]:
#Pull data (Google Sheets). Offline: load_raw_data(RAW_PATH) uses the last snapshot from build_features
df, flagged = clean_raw(load_raw_data())   # flagged = sets excluded by the bad-data guard
print(f"{len(df)} sets, {df['date'].nunique()} workouts ({df['date'].min().date()} to {df['date'].max().date()})")
df.dtypes

## Build features

Uses the same settings as the script. To experiment, override a copy, e.g. `settings = {**FEATURE_SETTINGS, 'metrics': ['avg', 'slope']}`.

In [ ]:
settings = dict(FEATURE_SETTINGS)
feats = F.build_features(df, **settings)
print(f'{len(df)} rows in -> {len(feats)} rows out, {feats.shape[1]} columns')
feats.tail()

## Sanity checks

In [ ]:
#Leakage spot-check: hand compute 30d history for one Smith Incline Bench set 2 row
sub = feats[(feats.exercise == 'Smith Incline Bench') & (feats.set_number == 2)].sort_values('date')
row = sub.iloc[len(sub) // 2]
d = row['date']
prior = sub[(sub.date < d) & (sub.date >= d - pd.Timedelta(days=30))]

print('date', d.date())
print('avg_30d  hand', round(prior.est_1rm.mean(), 3), '| feature', round(row.est_1rm_avg_30d, 3))
print('max_30d  hand', prior.est_1rm.max(), '| feature', row.est_1rm_max_30d)
print('most_recent hand', sub[sub.date < d].est_1rm.iloc[-1], '| feature', row.est_1rm_most_recent)
assert np.isclose(prior.est_1rm.mean(), row.est_1rm_avg_30d)

#First time an exercise + set appears it has no history
first = feats.sort_values('date').groupby(['exercise', 'set_number']).head(1)
assert first['est_1rm_most_recent'].isna().all()

In [ ]:
#Slope over time for one lift (lb of e1RM gained per day)
lift = feats[(feats.exercise == 'Smith Incline Bench') & (feats.set_number == 1)]
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(lift.date, lift.est_1rm, '.', label='est_1rm')
ax[0].plot(lift.date, lift.est_1rm_avg_30d, label='avg_30d')
ax[0].legend()
ax[1].plot(lift.date, lift.est_1rm_slope_90d, label='slope_90d')
ax[1].axhline(0, color='grey', lw=0.8)
ax[1].legend()
plt.show()

## Model inputs

`get_feature_columns` drops ids, same-session values (weight, reps, rpe, est_1rm, ratings, form_change_ind, comment), and raw current-day flags.

In [ ]:
feature_cols = F.get_feature_columns(feats)
print(len(feature_cols), 'features')
X = F.encode_categoricals(feats[feature_cols], method='category')
X.isna().mean().sort_values(ascending=False).head(15)

## Compare with the saved dataset

Checks that `data/features.parquet` (from the script) matches what this notebook builds from the same data.

In [ ]:
if DATA_PATH.exists():
    saved = pd.read_parquet(DATA_PATH)
    print('saved:', saved.shape, '| notebook:', feats.shape)
    print('same columns:', list(saved.columns) == list(feats.columns))
    print('saved data ends', pd.to_datetime(saved['date']).max().date(), '| sheet ends', df['date'].max().date())
else:
    print(f'{DATA_PATH} not built yet: run python -m modeling.build_features')